# Phase 4. Screen titles and abstracts

**Md Forhadul Islam**. Review of benchmarks for coding agents and agentic software engineering (2022 to 2026).

Records were screened on title and abstract with AI assisted screening under my written rules (`protocol/`). The
decisions are starting data in `screening/stage1_title_abstract/decisions/`. This notebook audits the keyword first pass,
merges the stage 1 decisions of the database search, Google Scholar and the first forward round into one list, counts them and draws the stage 1 PRISMA diagram.

In [ ]:
from setup import *
assert Path.cwd() == ROOT, "the notebook must work inside the Report folder"
print("Report folder:", ROOT)

The list below also shows the decision files of backward snowballing and of rounds 2 to 5. Script 05 does not read them: they enter the tracker and the yield per round later (notebooks 06 and 07).

In [ ]:
D1 = 'screening/stage1_title_abstract/decisions/'
rows = []
for f in sorted(glob.glob(str(ROOT / D1 / '*.json'))) + sorted(glob.glob(str(ROOT / 'snowballing/**/*_out.json'), recursive=True)):
    if f.endswith('stage1_all_decisions_merged.json'): continue
    d = json.load(open(f, encoding='utf-8'))
    rows.append((os.path.relpath(f, ROOT), len(d)))
import pandas as pd; display(pd.DataFrame(rows, columns=['decision file (starting data)', 'entries']))

## 4.1 Audit of the keyword first pass

Random samples of the records the keyword rule set aside were checked. The rule missed too many relevant records, so every record was screened one by one. The labels Exclude (automation) and Pending in Master_Tracker_v2 come from the keyword first pass and are replaced by the one by one decisions merged in script 05.

In [ ]:
au = J('screening/stage1_title_abstract/agreement_kappa/audit_auto_excluded_v1.json')
print(json.dumps({k: v for k, v in au.items() if not isinstance(v, list)}, indent=1)[:1500] if isinstance(au, dict) else len(au))

### `scripts/04_merge_stage1.py`

Merges the first screening decisions into the tracker and counts them per source.

In [ ]:
%%writefile scripts/04_merge_stage1.py
"""Merge stage 1 decisions (agents A, B for search; C for forward snowball) into Master_Tracker_v2.xlsx and recompute PRISMA counts."""
import json,sys,os,collections,openpyxl,re
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as ILL
clean=lambda v: ILL.sub("",v) if isinstance(v,str) else v
from openpyxl.styles import Font,PatternFill,Alignment
R=sys.argv[1]
fp=json.load(open(R+'/screening/stage1_title_abstract/decisions/auto_firstpass_v1.json'))
cls={r[0]:r[1] for r in fp}
dec={}
for f in ['agentA_decisions.json','agentB_decisions.json']:
    for d in json.load(open(R+'/screening/stage1_title_abstract/decisions/'+f)): dec[d['id']]=d
fwd=json.load(open(R+'/snowballing/round_1_forward/forward_new_records_with_ids.json',encoding='utf-8'))
fcls={r[0]:r[1] for r in json.load(open(R+'/snowballing/round_1_forward/auto_firstpass_forward_v1.json'))}
fdec={d['fid']:d for d in json.load(open(R+'/snowballing/round_1_forward/agentC_decisions.json'))}
have={}
for f in os.listdir(R+'/papers/pdfs/open_access'): have[f.split('_')[0]]='papers/pdfs/open_access/'+f
tx=json.load(open(R+'/papers/retrieval_log/text_index.json')) if os.path.exists(R+'/papers/retrieval_log/text_index.json') else {}
wb=openpyxl.load_workbook(R+'/tracker/master_tracker/Master_Tracker_v1.xlsx')
ws=wb['Master_Tracker'];h=[c.value for c in ws[1]];ix={n:i+1 for i,n in enumerate(h)}
ws.cell(1,ix['Auto priority (aid only)']).value='Auto first pass class';ix['Auto first pass class']=ix['Auto priority (aid only)']
for col,name in [(25,'Stage 1 code'),(26,'Task category'),(27,'Abstract from PDF (when available)')]: ws.cell(1,col).value=name
c=collections.Counter()
for row in range(2,ws.max_row+1):
    rid=ws.cell(row,1).value;k=cls.get(rid,'');ws.cell(row,ix['Auto first pass class']).value=k
    d=dec.get(rid)
    if d: ws.cell(row,ix['Stage 1 decision (Include/Exclude/Maybe)']).value=d['decision'];ws.cell(row,ix['Stage 1 reason']).value=d['reason'];ws.cell(row,ix['Benchmark name(s)']).value=d.get('benchmark_name');ws.cell(row,25).value=d['code'];ws.cell(row,26).value=d.get('task_category');c['S_'+d['decision']]+=1
    elif k in('Z_out_of_scope_auto',):ws.cell(row,ix['Stage 1 decision (Include/Exclude/Maybe)']).value='Exclude (automation)';c['S_auto_excluded']+=1
    elif k=='X_no_abstract':ws.cell(row,ix['Stage 1 decision (Include/Exclude/Maybe)']).value='Pending (no abstract, title check)';c['S_no_abstract']+=1
    if rid in have: ws.cell(row,ix['Full text saved (path)']).value=have[rid]
    if rid in tx and tx[rid].get('abstract_extracted'): ws.cell(row,27).value=clean(tx[rid]['abstract_extracted'])
# forward sheet
if 'Forward_Snowball_R1' in wb.sheetnames: del wb['Forward_Snowball_R1']
fs=wb.create_sheet('Forward_Snowball_R1')
hdr=['FID','Title','Authors','Year','Venue','Cited by','DOI','Open access','OA / PDF link','Cites seed(s)','Auto first pass class','Stage 1 decision','Stage 1 code','Reason','Benchmark name','Task category','Full text saved (path)','Abstract','OpenAlex ID']
fs.append(hdr)
for cc in fs[1]: cc.font=Font(bold=True,color='FFFFFF');cc.fill=PatternFill('solid',fgColor='375623')
fc=collections.Counter()
for r in fwd:
    d=fdec.get(r['fid']);k=fcls.get(r['fid'])
    sd=d['decision'] if d else ('Exclude (automation)' if k=='Z_out_of_scope_auto' else 'Pending (no abstract, title check)')
    fc[sd]+=1
    fs.append([clean(x) for x in [r['fid'],r['title'],r['authors'],r['year'],r['venue'],r['cites'],r['doi'],'yes' if r['oa'] else 'no',r['oa_url'],'; '.join(r['cites_seeds']),k,sd,d['code'] if d else '',d['reason'] if d else '',d.get('benchmark_name') if d else '',d.get('task_category') if d else '',have.get(r['fid'],''),r['abstract'],r['id']]])
for col,w in zip('ABCDEFGHIJKLMNOPQRS',[7,60,28,6,24,8,26,7,36,30,22,14,8,30,22,18,22,90,26]): fs.column_dimensions[col].width=w
fs.freeze_panes='C2';fs.auto_filter.ref=fs.dimensions
wb.save(R+'/tracker/master_tracker/Master_Tracker_v2.xlsx')
out={'search_stage1':dict(c),'forward_r1_stage1':dict(fc)}
json.dump(out,open(R+'/search/counts/prisma_counts_stage1_v2.json','w'),indent=1);print(out)

In [ ]:
p = run('04_merge_stage1.py')

### `scripts/05_stage1_merge_all.py`

Merges the stage 1 decisions of the database search, Google Scholar and the first forward round into one list and counts the records from the first round that go on to full text.

In [ ]:
%%writefile scripts/05_stage1_merge_all.py
"""Merge all stage 1 decisions from every source into one list and compute PRISMA stage 1 counts. Counts come only from files."""
import json,os,sys,collections,glob
R=sys.argv[1];D=R+'/screening/stage1_title_abstract/decisions/'
J=lambda p:json.load(open(p,encoding='utf-8'))
recs={}
# universe of screened records
raw=J(R+'/data/openalex_raw_2026-11-09.json')
fp=J(D+'auto_firstpass_v1.json')
for r in fp: recs[r[0]]={'id':r[0],'source':'database_search','title':r[2],'year':r[3],'decision':None}
fw=J(R+'/snowballing/round_1_forward/forward_new_records_with_ids.json')
for r in fw: recs[r['fid']]={'id':r['fid'],'source':'forward_snowball_seeds9','title':r['title'],'year':r['year'],'decision':None}
sa=J(R+'/snowballing/round_1_forward/sweagent_new_records_with_ids.json')
for r in sa: recs[r['sid']]={'id':r['sid'],'source':'forward_snowball_sweagent','title':r['title'],'year':r['year'],'decision':None}
gs=J(D+'scholar_only_decisions.json')
for d in gs: recs[d['id']]={'id':d['id'],'source':'google_scholar','title':d['title'],'year':None,'decision':None}
def setd(i,d,src):
    if i not in recs: return
    recs[i].update({'decision':d.get('decision'),'code':d.get('code'),'benchmark_name':d.get('benchmark_name'),'reason':d.get('reason'),'task_category':d.get('task_category'),'decided_by':src})
for d in J(D+'agentA_decisions.json'): setd(d['id'],d,'screener_A')
for d in J(D+'agentB_decisions.json'): setd(d['id'],d,'screener_B')
for d in J(R+'/snowballing/round_1_forward/agentC_decisions.json'): setd(d['fid'],d,'screener_C')
for d in gs: setd(d['id'],d,'scholar_screener')
au=J(R+'/screening/stage1_title_abstract/agreement_kappa/audit_auto_excluded_v1.json')
for d in au['sample_search']: setd(d['id'],d,'audit')
for d in au['sample_forward']: setd(d['fid'],d,'audit')
for d in au['no_abstract']: setd(d['id_or_fid'],d,'audit_title_only')
for k in range(1,7):
    o=J(D+f'fullscreen_batch_{k}_out.json')
    for i in o['reviewed_ids']:
        if i in recs and recs[i]['decision'] is None: recs[i].update({'decision':'Exclude','decided_by':f'fullscreen_{k}'})
    for d in o['include_or_maybe']: setd(d['id'],d,f'fullscreen_{k}')
for r in recs.values():
    if r['decision'] is None: r['decision']='UNSCREENED'
    if r['decision'].startswith('Exclude'): r['decision']='Exclude'
c=collections.Counter((r['source'],r['decision']) for r in recs.values())
tot=collections.Counter(r['decision'] for r in recs.values())
inc=[r for r in recs.values() if r['decision'] in('Include','Maybe')]
json.dump(list(recs.values()),open(D+'stage1_all_decisions_merged.json','w'),indent=0)
summary={'by_source_and_decision':{f'{a}|{b}':v for (a,b),v in sorted(c.items())},'totals':dict(tot),'to_full_text':len(inc)}
json.dump(summary,open(R+'/search/counts/prisma_stage1_final_v1.json','w'),indent=1)
print(json.dumps(summary,indent=1))

In [ ]:
p = run('05_stage1_merge_all.py')
s = J('search/counts/prisma_stage1_final_v1.json'); expect('records sent to full text from the first round', s['to_full_text'], 1220)
same_as_original('search/counts/prisma_stage1_final_v1.json'); same_as_original('screening/stage1_title_abstract/decisions/stage1_all_decisions_merged.json')

### `scripts/06_prisma_flow_stage1.py`

Draws the interim PRISMA diagram after stage 1.

In [ ]:
%%writefile scripts/06_prisma_flow_stage1.py
"""PRISMA 2020 flow diagram (stage 1 status). Every number is read from saved files."""
import json,sys,os
import matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
R=sys.argv[1];J=lambda p:json.load(open(p,encoding='utf-8'))
raw=J(R+'/data/openalex_raw_2026-11-09.json');gsr=J(R+'/data/scholar_raw_2026-11-09.json')
c1=J(R+'/search/counts/prisma_counts_v1.json');so=J(R+'/search/counts/scholar_overlap_v1.json')
fwd=J(R+'/data/openalex_forward_snowball_round1_2026-11-09.json');sa=J(R+'/data/openalex_forward_sweagent_and_seeds_2026-11-09.json')
fnew=len(J(R+'/snowballing/round_1_forward/forward_new_records_with_ids.json'));snew=len(J(R+'/snowballing/round_1_forward/sweagent_new_records_with_ids.json'))
s=J(R+'/search/counts/prisma_stage1_final_v1.json')['by_source_and_decision']
g=lambda k:s.get(k,0)
n_oa=len(raw['records']);n_gs=len(gsr['records']);n_f=len(fwd['records']);n_sa=len(sa['records']);n_seed=3
dup_db=c1['duplicates_removed']+(n_gs-so['new'])
dup_ot=(n_f-fnew)+(n_sa-snew)
scr_db=c1['unique_records']+so['new'];scr_ot=fnew+snew
exc_db=g('database_search|Exclude')+g('google_scholar|Exclude');exc_ot=g('forward_snowball_seeds9|Exclude')+g('forward_snowball_sweagent|Exclude')
sought_db=scr_db-exc_db;sought_ot=scr_ot-exc_ot+n_seed
fig,ax=plt.subplots(figsize=(13,11));ax.set_xlim(0,100);ax.set_ylim(0,100);ax.axis('off')
def box(x,y,w,h,t,fc='white'):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.3',fc=fc,ec='black',lw=1.1));ax.text(x+w/2,y+h/2,t,ha='center',va='center',fontsize=8.6,wrap=True)
def arr(x1,y1,x2,y2): ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='->',lw=1.1))
ax.text(25,97,'Identification of studies via databases',ha='center',fontsize=11,weight='bold')
ax.text(76,97,'Identification of studies via other methods',ha='center',fontsize=11,weight='bold')
box(2,80,40,13,f'Records identified (Nov 2022 to Nov 2026):\nOpenAlex, 5 queries (n = {n_oa:,})\nGoogle Scholar, top 50 of 5 queries (n = {n_gs:,})')
box(56,80,42,13,f'Records identified from:\nForward citations of 9 seed benchmarks (n = {n_f:,})\nForward citations of SWE-agent (n = {n_sa:,})\nSeed papers added directly (n = {n_seed})')
box(2,62,40,12,f'Duplicates removed before screening\n(n = {dup_db:,})',fc='#f2f2f2')
box(56,62,42,12,f'Records already found by database search\nor duplicates removed (n = {dup_ot:,})',fc='#f2f2f2')
box(2,44,40,12,f'Records screened (title and abstract)\n(n = {scr_db:,})')
box(56,44,42,12,f'Records screened (title and abstract)\n(n = {scr_ot:,}) plus {n_seed} seed papers')
box(2,26,40,12,f'Records excluded at title and abstract\n(n = {exc_db:,})',fc='#f2f2f2')
box(56,26,42,12,f'Records excluded at title and abstract\n(n = {exc_ot:,})',fc='#f2f2f2')
box(2,6,40,14,f'Reports sought for full text\n(n = {sought_db:,}; {g("database_search|Include")+g("google_scholar|Include")} include, {g("database_search|Maybe")+g("google_scholar|Maybe")} maybe)\nFull text assessment: in progress',fc='#e8f0fe')
box(56,6,42,14,f'Reports sought for full text\n(n = {sought_ot:,}; {g("forward_snowball_seeds9|Include")+g("forward_snowball_sweagent|Include")+n_seed} include, {g("forward_snowball_seeds9|Maybe")+g("forward_snowball_sweagent|Maybe")} maybe)\nBackward snowballing: next step',fc='#e8f0fe')
for x in (22,77):
    arr(x,80,x,74.6);arr(x,62,x,56.6);arr(x,44,x,38.6);arr(x,26,x,20.6)
ax.text(50,1,'Stage 1 status, 9 November 2026. Generated by scripts/06_prisma_flow_stage1.py from saved search and screening files.',ha='center',fontsize=7.5,style='italic')
os.makedirs(R+'/figures/prisma_flow_diagram',exist_ok=True)
for ext in ('png','pdf'): plt.savefig(R+f'/figures/prisma_flow_diagram/PRISMA_flow_stage1_v1.{ext}',dpi=200,bbox_inches='tight')
print(dict(n_oa=n_oa,n_gs=n_gs,n_f=n_f,n_sa=n_sa,dup_db=dup_db,dup_ot=dup_ot,scr_db=scr_db,scr_ot=scr_ot,exc_db=exc_db,exc_ot=exc_ot,sought_db=sought_db,sought_ot=sought_ot))

In [ ]:
p = run('06_prisma_flow_stage1.py')
s = J('search/counts/prisma_stage1_final_v1.json')
print(f"{s['to_full_text']} records were sent to full text in the first round; without the two Scholar only duplicates GS028 and GS051 the figure is {s['to_full_text'] - 2}")

These interim figures still use the 74 Scholar only records of the first matching pass. Five of them (GS016, GS028, GS030, GS051, GS058) are duplicates of OpenAlex records; GS028 (Include) and GS051 (Maybe) are among the 1,220 records sent to full text. They are removed in the tracker (`07_build_master_tracker.py`) and in the final flow diagram, which use `scholar_overlap_v2`.

### `scripts/07_task_category_tree.py`

First grouping of the task labels noted during screening (the start of the taxonomy).

In [ ]:
%%writefile scripts/07_task_category_tree.py
"""Stage 1 keywording and clustering of task_category labels (Include and Maybe records) into a two level tree.
Counts come only from stage1_all_decisions_merged.json. Rules are ordered; the first rule that matches wins."""
import json,sys,collections,re
R=sys.argv[1]
m=json.load(open(R+'/screening/stage1_title_abstract/decisions/stage1_all_decisions_merged.json',encoding='utf-8'))
im=[r for r in m if r['decision'] in ('Include','Maybe')]
L='Meta research on benchmarks'
RULES=[
 # (regex on label, level 1, level 2)
 (r'survey|review note', 'Surveys and reviews of benchmarks','Surveys and reviews'),
 (r'contamination', L,'Contamination and memorisation analysis'),
 (r'validity|integrity|reward hacking|saturation|critique|quality analysis', L,'Validity, integrity and saturation'),
 (r'analysis|evaluation methodology|cost', L,'Benchmark analyses and evaluation methodology'),
 (r'benchmark infrastructure|benchmark framework|evaluation infrastructure|agent harness|code evaluation|repository construction', L,'Benchmark construction infrastructure'),
 (r'security|safety|honesty|trustworth|compliance', 'Security, safety and trust','Security and safety'),
 (r'scientific|ml |ml$|machine learning|data science|data engineering|research cod|ml research|robotics', 'Data science, ML and scientific code',None),
 (r'hardware|embedded|compiler|game|model based|kernel|automation engineering|systems design|verified systems|systems se', 'Specialised engineering domains','Hardware, embedded, systems and other domains'),
 (r'issue locali|fault locali|code locali|repository navigation', 'Issue resolution and program repair','Issue and fault localisation'),
 (r'program repair|debugging|bug repair|bug fixing', 'Issue resolution and program repair','Program repair and debugging'),
 (r'issue', 'Issue resolution and program repair','Issue resolution'),
 (r'devops|terminal|cli|sre|aiops|cloud|build|environment setup|environment and dependency|ci repair|dependency upgrade|infrastructure building', 'Environment, terminal and DevOps tasks','DevOps, build and terminal'),
 (r'agent environment|interactive coding|tool using', 'Environment, terminal and DevOps tasks','General agent environments and gyms'),
 (r'test|code review|verification|bug discovery', 'Testing, review and verification',None),
 (r'refactor|cleanup|smell|optimi', 'Maintenance and evolution','Refactoring and optimisation'),
 (r'migration|backport|translation', 'Maintenance and evolution','Migration and porting'),
 (r'maintenance|evolution|merge|version control|documentation', 'Maintenance and evolution','Maintenance, merging and evolution'),
 (r'web|app|game|end-to-end|project generation|artifact building|software building|application', 'Feature and application development','Web and application building'),
 (r'feature|long.horizon|library design|software design', 'Feature and application development','Feature implementation and long horizon work'),
 (r'completion', 'Repository level code generation and completion','Repository level completion'),
 (r'generation|repository level|repository code|code editing|library generation|repository context|competitive|unfamiliar', 'Repository level code generation and completion','Repository level generation and coding'),
 (r'reasoning|understanding|comprehension|qa$|requirement|specification|instruction', 'Code understanding, requirements and instructions',None),
 (r'multi agent|collaborat|multi language|multilingual|multimodal|multi task|general se|agent software engineering', 'Cross cutting coverage (multi language, multimodal, multi agent)',None),
 (r'agent', 'Agent behaviour and usage studies',None),
]
SUB={ # level 2 for rules whose level 2 is None
 'Data science, ML and scientific code':[(r'scientific|research cod|robotics','Scientific and research code'),(r'ml|machine','ML engineering'),(r'data','Data science and data engineering')],
 'Testing, review and verification':[(r'review','Code review'),(r'verification','Formal verification'),(r'test|bug discovery','Test generation and testing')],
 'Code understanding, requirements and instructions':[(r'requirement|specification','Requirements and specification'),(r'instruction','Instruction following'),(r'.','Code reasoning and repository QA')],
 'Cross cutting coverage (multi language, multimodal, multi agent)':[(r'multi language|multilingual','Multi language'),(r'multimodal','Multimodal'),(r'multi agent|collaborat','Multi agent collaboration'),(r'.','Multi task and general SE')],
 'Agent behaviour and usage studies':[(r'pull request|usage|real world','Agent pull requests and usage logs'),(r'.','Agent behaviour, skills and tooling')],
}
def classify(lab):
    if lab in ('','other','unclear','na','general agent incl coding'): return ('Unclassified at stage 1','Other or no label')
    for pat,l1,l2 in RULES:
        if re.search(pat,lab):
            if l2 is None:
                for p2,s in SUB[l1]:
                    if re.search(p2,lab): return (l1,s)
            return (l1,l2)
    return ('Unclassified at stage 1','Other or no label')
tree=collections.defaultdict(collections.Counter); labmap={}
for r in im:
    lab=(r.get('task_category') or '').strip().lower()
    c=classify(lab); labmap[lab]=c; tree[c[0]][c[1]]+=1
out={'n_records':len(im),'level1':{k:sum(v.values()) for k,v in sorted(tree.items(),key=lambda x:-sum(x[1].values()))},
     'level2':{k:dict(v.most_common()) for k,v in tree.items()},
     'label_to_category':{k:list(v) for k,v in sorted(labmap.items())}}
# benchmark names (normalised) and contribution codes
norm=lambda s:re.sub(r'[\s\-_]','',s.lower())
bn=collections.Counter(); disp={}
for r in im:
    b=(r.get('benchmark_name') or '').strip()
    if not b or b.upper()=='NA': continue
    k=norm(b); bn[k]+=1; disp.setdefault(k,b)
out['benchmark_names']={'records_with_name':sum(bn.values()),'distinct_names':len(bn),'top':[[disp[k],v] for k,v in bn.most_common(25)]}
out['contribution_codes']=dict(collections.Counter((r.get('code') or 'none') for r in im))
out['decisions']=dict(collections.Counter(r['decision'] for r in im))
json.dump(out,open(R+'/taxonomy/data/task_category_tree_stage1_v1.json','w',encoding='utf-8'),indent=1,ensure_ascii=False)
print(json.dumps({k:out[k] for k in ('n_records','level1','level2','benchmark_names','contribution_codes','decisions')},indent=1,ensure_ascii=False))

In [ ]:
p = run('07_task_category_tree.py')
same_as_original('taxonomy/data/task_category_tree_stage1_v1.json')

In [ ]:
save_checks('04_Title_abstract_screening')